***Problem Selection and Domain Identification***
-

*Project Title*

“Banking Customer Complaint Information Extraction System Using NLP”

*Objective*

The objective of this project is to develop an NLP-based Information Extraction System that processes banking-related customer queries and identifies useful information from the given text.

The system will use text preprocessing and rule-based NLP techniques to extract relevant entities or information and present them in a structured format.

**Loading the dataset**

In [2]:
import pandas as pd

df = pd.read_csv("dev.csv")

print("===== DATASET INFORMATION =====")

print("Number of records:", len(df))
print("Number of columns:", len(df.columns))

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 records:")
print(df.head())

print("\nDataset shape:")
print(df.shape)

===== DATASET INFORMATION =====
Number of records: 3080
Number of columns: 3

Column names:
['text', 'label', 'label_text']

First 5 records:
                                                text  label    label_text
0                           How do I locate my card?     11  card_arrival
1  I still have not received my new card, I order...     11  card_arrival
2  I ordered a card but it has not arrived. Help ...     11  card_arrival
3   Is there a way to know when my card will arrive?     11  card_arrival
4                       My card has not arrived yet.     11  card_arrival

Dataset shape:
(3080, 3)


**Dataset Exploration and Understanding**

In [3]:
print("===== DATASET SHAPE =====")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


print("\n===== COLUMN NAMES =====")
print(df.columns.tolist())


print("\n===== FIRST 5 RECORDS =====")
print(df.head())


print("\n===== DATA TYPES =====")
print(df.dtypes)


print("\n===== MISSING VALUES =====")
print(df.isnull().sum())


print("\n===== DUPLICATE RECORDS =====")
print("Number of duplicate records:", df.duplicated().sum())


print("\n===== NUMBER OF CATEGORIES =====")
print("Number of unique categories:", df["label_text"].nunique())


print("\n===== CATEGORY NAMES =====")
print(df["label_text"].unique())


print("\n===== CATEGORY DISTRIBUTION =====")
print(df["label_text"].value_counts())


print("\n===== SAMPLE CUSTOMER QUERIES =====")
for i, text in enumerate(df["text"].head(10), start=1):
    print(f"{i}. {text}")

print("===== DATASET SHAPE =====")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


print("\n===== COLUMN NAMES =====")
print(df.columns.tolist())


print("\n===== FIRST 5 RECORDS =====")
print(df.head())


print("\n===== DATA TYPES =====")
print(df.dtypes)


print("\n===== MISSING VALUES =====")
print(df.isnull().sum())


print("\n===== DUPLICATE RECORDS =====")
print("Number of duplicate records:", df.duplicated().sum())


print("\n===== NUMBER OF CATEGORIES =====")
print("Number of unique categories:", df["label_text"].nunique())


print("\n===== CATEGORY NAMES =====")
print(df["label_text"].unique())


print("\n===== CATEGORY DISTRIBUTION =====")
print(df["label_text"].value_counts())


print("\n===== SAMPLE CUSTOMER QUERIES =====")
for i, text in enumerate(df["text"].head(10), start=1):
    print(f"{i}. {text}")

===== DATASET SHAPE =====
Rows: 3080
Columns: 3

===== COLUMN NAMES =====
['text', 'label', 'label_text']

===== FIRST 5 RECORDS =====
                                                text  label    label_text
0                           How do I locate my card?     11  card_arrival
1  I still have not received my new card, I order...     11  card_arrival
2  I ordered a card but it has not arrived. Help ...     11  card_arrival
3   Is there a way to know when my card will arrive?     11  card_arrival
4                       My card has not arrived yet.     11  card_arrival

===== DATA TYPES =====
text          object
label          int64
label_text    object
dtype: object

===== MISSING VALUES =====
text          0
label         0
label_text    0
dtype: int64

===== DUPLICATE RECORDS =====
Number of duplicate records: 0

===== NUMBER OF CATEGORIES =====
Number of unique categories: 77

===== CATEGORY NAMES =====
['card_arrival' 'card_linking' 'exchange_rate'
 'card_payment_wrong_exchang

**Inference:** *The dataset contains 3,080 banking-related customer queries belonging to 77 different categories. Each category contains 40 records, making the dataset balanced. There are no missing values or duplicate records, so the dataset is suitable for further NLP processing. The text column will be used as the main input for our Information Extraction System, while label_text can help us understand the type of banking-related query.*

**Text Preprocessing**

In [4]:
import re
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lapto\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lapto\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lapto\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
df["clean_text"] = df["text"].copy()

stop_words = set(stopwords.words("english"))

def preprocess_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    tokens = word_tokenize(text)

    tokens = [word for word in tokens if word not in stop_words]

    return ' '.join(tokens)


df["clean_text"] = df["text"].apply(preprocess_text)


print("===== ORIGINAL AND PREPROCESSED TEXT =====")

for i in range(5):
    print("\nOriginal :", df["text"].iloc[i])
    print("Cleaned  :", df["clean_text"].iloc[i])

===== ORIGINAL AND PREPROCESSED TEXT =====

Original : How do I locate my card?
Cleaned  : locate card

Original : I still have not received my new card, I ordered over a week ago.
Cleaned  : still received new card ordered week ago

Original : I ordered a card but it has not arrived. Help please!
Cleaned  : ordered card arrived help please

Original : Is there a way to know when my card will arrive?
Cleaned  : way know card arrive

Original : My card has not arrived yet.
Cleaned  : card arrived yet


**Inference:** *The text preprocessing step converted the customer queries into a cleaner and more consistent format. Lowercase conversion, punctuation removal, extra-space removal, tokenization, and stopword removal were performed. The resulting clean_text column contains the processed text, which can be used for the next stage of the project, Information Extraction.*

**Information Extraction**

In [6]:
import re

def extract_information(text):

    result = {
        "Amount": "Not Mentioned",
        "Payment/Transaction Type": "Not Mentioned",
        "Transaction Status": "Not Mentioned",
        "Card Information": "Not Mentioned",
        "Banking Service": "Not Mentioned",
        "Issue": "Not Mentioned"
    }

    text_lower = text.lower()

    amount_pattern = r'(?:₹|rs\.?|inr|\$|€|£)\s?\d+(?:[.,]\d+)?|\d+(?:[.,]\d+)?\s?(?:rupees|rs|inr)'

    amount = re.search(amount_pattern, text_lower)

    if amount:
        result["Amount"] = amount.group()

    payment_types = {
        "upi": ["upi"],
        "bank transfer": ["bank transfer", "transfer"],
        "card payment": ["card payment", "card purchase", "paid by card"],
        "cash withdrawal": ["cash withdrawal", "withdrew", "withdrawal"],
        "cash deposit": ["cash deposit", "deposited cash"],
        "cash transfer": ["cash transfer"],
        "top up": ["top up", "top-up", "topped up"],
        "direct debit": ["direct debit"],
        "contactless payment": ["contactless"],
    }

    for payment_type, keywords in payment_types.items():

        if any(keyword in text_lower for keyword in keywords):
            result["Payment/Transaction Type"] = payment_type
            break


    status_keywords = {
        "Pending": [
            "pending",
            "still waiting",
            "waiting",
            "not completed"
        ],

        "Failed": [
            "failed",
            "failure",
            "didn't work",
            "does not work",
            "not working"
        ],

        "Successful": [
            "successful",
            "successfully",
            "completed"
        ],

        "Declined": [
            "declined",
            "rejected"
        ],

        "Cancelled": [
            "cancelled",
            "canceled"
        ],

        "Refunded": [
            "refunded",
            "refund received",
            "got my refund"
        ]
    }

    for status, keywords in status_keywords.items():

        if any(keyword in text_lower for keyword in keywords):
            result["Transaction Status"] = status
            break


    card_keywords = {
        "Card Arrival": [
            "card has not arrived",
            "card hasn't arrived",
            "card arrival",
            "card delivery",
            "card arrive"
        ],

        "Lost/Stolen Card": [
            "lost card",
            "stolen card",
            "card was stolen",
            "lost my card"
        ],

        "Card Not Working": [
            "card not working",
            "card doesn't work",
            "card does not work"
        ],

        "Card Payment": [
            "card payment",
            "paid by card",
            "payment with my card"
        ],

        "Contactless": [
            "contactless",
            "tap to pay"
        ],

        "PIN Problem": [
            "pin",
            "pin blocked",
            "forgot my pin"
        ]
    }

    for card_type, keywords in card_keywords.items():

        if any(keyword in text_lower for keyword in keywords):
            result["Card Information"] = card_type
            break


    banking_services = {
        "Card": ["card", "visa", "mastercard"],
        "Transfer": ["transfer", "transferred"],
        "Cash Withdrawal": ["cash withdrawal", "cash machine", "atm"],
        "Cash Deposit": ["cash deposit", "deposit"],
        "Top Up": ["top up", "top-up"],
        "Refund": ["refund", "refunded"],
        "Exchange": ["exchange rate", "currency exchange", "exchange"],
        "Direct Debit": ["direct debit"],
        "Account": ["account", "bank account"]
    }

    for service, keywords in banking_services.items():

        if any(keyword in text_lower for keyword in keywords):
            result["Banking Service"] = service
            break


    issues = {
        "Card Not Received": [
            "card has not arrived",
            "card hasn't arrived",
            "card not received",
            "card has not been received"
        ],

        "Payment Failed": [
            "payment failed",
            "payment failure",
            "payment didn't work",
            "payment does not work"
        ],

        "Payment Pending": [
            "payment pending",
            "payment is pending",
            "transaction pending"
        ],

        "Money Not Received": [
            "money has not arrived",
            "money not received",
            "did not receive the money",
            "haven't received the money"
        ],

        "Card Not Working": [
            "card not working",
            "card doesn't work",
            "card does not work"
        ],

        "Wrong Exchange Rate": [
            "wrong exchange rate",
            "incorrect exchange rate",
            "bad exchange rate"
        ],

        "Extra Charge": [
            "extra charge",
            "charged extra",
            "additional charge"
        ],

        "Refund Problem": [
            "refund",
            "refund not received",
            "waiting for refund"
        ],

        "Lost or Stolen Card": [
            "lost card",
            "stolen card",
            "lost my card"
        ]
    }

    for issue, keywords in issues.items():

        if any(keyword in text_lower for keyword in keywords):
            result["Issue"] = issue
            break


    return result

df["extracted_information"] = df["text"].apply(
    extract_information
)

print("===== EXTRACTED INFORMATION =====")

for i in range(10):

    print("\nCustomer Query:")
    print(df["text"].iloc[i])

    print("\nExtracted Information:")
    for key, value in df["extracted_information"].iloc[i].items():
        print(f"{key}: {value}")

    print("-" * 60)

===== EXTRACTED INFORMATION =====

Customer Query:
How do I locate my card?

Extracted Information:
Amount: Not Mentioned
Payment/Transaction Type: Not Mentioned
Transaction Status: Not Mentioned
Card Information: Not Mentioned
Banking Service: Card
Issue: Not Mentioned
------------------------------------------------------------

Customer Query:
I still have not received my new card, I ordered over a week ago.

Extracted Information:
Amount: Not Mentioned
Payment/Transaction Type: Not Mentioned
Transaction Status: Not Mentioned
Card Information: Not Mentioned
Banking Service: Card
Issue: Not Mentioned
------------------------------------------------------------

Customer Query:
I ordered a card but it has not arrived. Help please!

Extracted Information:
Amount: Not Mentioned
Payment/Transaction Type: Not Mentioned
Transaction Status: Not Mentioned
Card Information: Not Mentioned
Banking Service: Card
Issue: Not Mentioned
------------------------------------------------------------

C

**Display Extracted Information in Structured Format**

In [7]:
extracted_df = pd.json_normalize(df["extracted_information"])

extracted_df.insert(0, "Customer Query", df["text"].values)

print("===== STRUCTURED EXTRACTED INFORMATION =====")

extracted_df.head(10)

===== STRUCTURED EXTRACTED INFORMATION =====


,Customer Query,Amount,Payment/Transaction Type,Transaction Status,Card Information,Banking Service,Issue
0,How do I locate my card?,Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned,Card,Not Mentioned
1,"I still have not received my new card, I order...",Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned,Card,Not Mentioned
2,I ordered a card but it has not arrived. Help ...,Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned,Card,Not Mentioned
3,Is there a way to know when my card will arrive?,Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned,Card,Not Mentioned
4,My card has not arrived yet.,Not Mentioned,Not Mentioned,Not Mentioned,Card Arrival,Card,Card Not Received
5,When will I get my card?,Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned,Card,Not Mentioned
6,Do you know if there is a tracking number for ...,Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned,Card,Not Mentioned
7,i have not received my card,Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned,Card,Not Mentioned
8,still waiting on that card,Not Mentioned,Not Mentioned,Pending,Not Mentioned,Card,Not Mentioned
9,Is it normal to have to wait over a week for m...,Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned,Card,Not Mentioned


**Inference:** *The extracted information was converted from dictionary format into a structured table using separate columns for each information field. This makes the extracted banking information easier to read, analyze, and store. The structured results were also saved as a CSV file for further use.*

**Testing the Information Extraction System**

In [8]:
test_inputs = [
    "I made a payment of Rs 2500 using my card but the payment was declined.",
    
    "My card has not arrived yet. I ordered it last week.",
    
    "I tried to transfer money but the transaction is still pending.",
    
    "I lost my card yesterday and I want to block it.",
    
    "I was charged an extra fee during my card payment.",
    
    "I am waiting for my refund but I have not received it yet."

    "I tried to withdraw ₹5000 from ATM but it was declined."
]

for i, complaint in enumerate(test_inputs, start=1):

    print("\n" + "=" * 70)
    print(f"TEST CASE {i}")
    print("=" * 70)

    print("Input:")
    print(complaint)

    result = extract_information(complaint)

    print("\nExtracted Information:")

    for key, value in result.items():
        print(f"{key}: {value}")


TEST CASE 1
Input:
I made a payment of Rs 2500 using my card but the payment was declined.

Extracted Information:
Amount: rs 2500
Payment/Transaction Type: Not Mentioned
Transaction Status: Declined
Card Information: Not Mentioned
Banking Service: Card
Issue: Not Mentioned

TEST CASE 2
Input:
My card has not arrived yet. I ordered it last week.

Extracted Information:
Amount: Not Mentioned
Payment/Transaction Type: Not Mentioned
Transaction Status: Not Mentioned
Card Information: Card Arrival
Banking Service: Card
Issue: Card Not Received

TEST CASE 3
Input:
I tried to transfer money but the transaction is still pending.

Extracted Information:
Amount: Not Mentioned
Payment/Transaction Type: bank transfer
Transaction Status: Pending
Card Information: Not Mentioned
Banking Service: Transfer
Issue: Not Mentioned

TEST CASE 4
Input:
I lost my card yesterday and I want to block it.

Extracted Information:
Amount: Not Mentioned
Payment/Transaction Type: Not Mentioned
Transaction Status: N

**Inference:** *The Information Extraction System was tested using multiple banking-related customer queries. The system successfully identified relevant information such as transaction amount, transaction type, transaction status, card-related information, banking service, and complaint issue. The testing demonstrates that the rule-based NLP system can convert unstructured customer queries into structured information.*

### Conclusion

The **Banking Customer Complaint Information Extraction System Using NLP** was successfully developed using a rule-based approach. The system takes banking-related customer queries as input, performs text preprocessing, and extracts useful information such as transaction amount, transaction type, transaction status, card information, banking service, and complaint issue. The extracted information is then displayed in a structured format.

The system was tested with multiple sample inputs and dataset records. The project demonstrates how Natural Language Processing can be used to convert unstructured customer text into structured information. Although the current system is based on predefined rules and has some limitations, it provides a simple and effective solution for a small-scale banking information extraction application.